This notebook establishes the canonical baseline for Empathy prediction. Following the multi-target exploration in 02_baseline_multi_target.ipynb, we focus on a single target — Empathy — using TF-IDF features and Ridge regression on the same training data. We train on the training split, then evaluate on both dev and test to establish reference numbers used throughout the rest of the thesis.

In [2]:
!git clone https://github.com/DavorSopar/dataset-analysis.git /content/dataset-analysis
import sys
sys.path.insert(0, '/content/dataset-analysis/src')

fatal: destination path '/content/dataset-analysis' already exists and is not an empty directory.


In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score

# Make src/ importable whether run from notebooks/ or the repo root.
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src").is_dir():
    pass
elif (REPO_ROOT.parent / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from data import load_convt, impute_selfdisclosure, TARGETS

RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

In [4]:
# Load and resolve the two missing SelfDisclosure values (speaker-mean; see 01_eda).
df = impute_selfdisclosure(load_convt())
print(f"{len(df):,} turns across {df['conversation_id'].nunique()} conversations")
print("Targets:", TARGETS)

11,166 turns across 487 conversations
Targets: ['Emotion', 'EmotionalPolarity', 'Empathy', 'SelfDisclosure']


In [5]:
groups = df["conversation_id"].to_numpy()

# 70% train, 30% temp
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_idx, temp_idx = next(gss1.split(df, groups=groups))

# split temp 50/50 -> 15% dev, 15% test (still grouped)
temp = df.iloc[temp_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
dev_rel, test_rel = next(gss2.split(temp, groups=temp["conversation_id"].to_numpy()))
dev_idx, test_idx = temp_idx[dev_rel], temp_idx[test_rel]

train, dev, test = df.iloc[train_idx], df.iloc[dev_idx], df.iloc[test_idx]

# Guarantee no conversation appears in more than one split.
assert not (set(train.conversation_id) & set(dev.conversation_id))
assert not (set(train.conversation_id) & set(test.conversation_id))
assert not (set(dev.conversation_id) & set(test.conversation_id))

split_tbl = pd.DataFrame({
    "turns": [len(train), len(dev), len(test)],
    "conversations": [train.conversation_id.nunique(),
                      dev.conversation_id.nunique(),
                      test.conversation_id.nunique()],
}, index=["train", "dev", "test"])
split_tbl["turns_%"] = (100 * split_tbl["turns"] / len(df)).round(1)
print(split_tbl)
print("\nNo conversation overlaps across splits. Good.")

       turns  conversations  turns_%
train   7788            340  69.7000
dev     1640             73  14.7000
test    1738             74  15.6000

No conversation overlaps across splits. Good.


In [6]:
def pearson(y_true, y_pred):
    """Pearson r; returns NaN when either side is constant (undefined)."""
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if np.std(y_true) < 1e-12 or np.std(y_pred) < 1e-12:
        return np.nan
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def score_all(y_true, y_pred):
    return {
        "pearson": pearson(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

In [7]:
# Fit the vectorizer on TRAIN text only, then transform every split with it.
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_train = vectorizer.fit_transform(train["text"])
X_dev = vectorizer.transform(dev["text"])
X_test = vectorizer.transform(test["text"])
feature_names = np.array(vectorizer.get_feature_names_out())
print("TF-IDF matrix:", X_train.shape, "(train)")
print("Vocabulary size:", len(feature_names))

TF-IDF matrix: (7788, 5000) (train)
Vocabulary size: 5000


In [8]:
BEST_ALPHA = 3.0   # from Day 3 sweep
TARGET = 'Empathy'

y_train_e = train[TARGET]
y_dev_e = dev[TARGET]
y_test_e = test[TARGET]

model = Ridge(alpha=BEST_ALPHA).fit(X_train, y_train_e)

def evaluate(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    r = np.corrcoef(y_true, y_pred)[0, 1]
    print(f"{label:25s}  MAE={mae:.4f}  RMSE={rmse:.4f}  Pearson r={r:.4f}")
    return {'label': label, 'mae': mae, 'rmse': rmse, 'pearson': r}
results = [
    evaluate(y_train_e, model.predict(X_train), "Baseline Ridge · train"),
    evaluate(y_dev_e,   model.predict(X_dev),   "Baseline Ridge · dev"),
    evaluate(y_test_e,  model.predict(X_test),  "Baseline Ridge · test"),
]

Baseline Ridge · train     MAE=0.4715  RMSE=0.5934  Pearson r=0.7732
Baseline Ridge · dev       MAE=0.5655  RMSE=0.7104  Pearson r=0.6396
Baseline Ridge · test      MAE=0.6062  RMSE=0.7569  Pearson r=0.6442


In [10]:
import joblib

# Save results
pd.DataFrame(results).to_csv('day05_baseline_empathy_results.csv', index=False)

# Save trained model and vectorizer
joblib.dump(model, 'baseline_ridge_empathy.pkl')
joblib.dump(vectorizer, 'baseline_vectorizer_empathy.pkl')

print("Saved: day05_baseline_empathy_results.csv")
print("Saved: baseline_ridge_empathy.pkl")
print("Saved: baseline_vectorizer_empathy.pkl")

Saved: day05_baseline_empathy_results.csv
Saved: baseline_ridge_empathy.pkl
Saved: baseline_vectorizer_empathy.pkl
